# Score the 2020-2024 GKG backfill from CSVs (Kaggle GPU)

Scores the backtest corpus that moved out of Supabase into files. Input is
the `gkg_2020.csv` .. `gkg_2024.csv` files produced by
`python -m pipeline.backfill_gkg --year <y> --to-csv <dir>` (~400k rows,
columns `bank_id, published_at, title, url, language, attributed`). Output
is ONE parquet in `/kaggle/working` — no database access at all.

Rows are filtered exactly the way serving does (`pipeline.eligibility`, the
gdelt adapter: English, non-empty title), then **everything eligible is
scored, including unattributed rows** — the backtest may tune the
attribution gate, and re-scoring is cheap on GPU. Ineligible rows are simply
absent from the parquet; the funnel counts are printed instead.

## Before running

1. Settings → Accelerator → **GPU**. Internet can stay **Off** — nothing
   here dials out.
2. Add Data → attach **`chloejiwon/finbert-ft-2026-08-09`** (the weights).
3. Add Data → attach the dataset holding the backfill CSVs.
4. Add Data → attach the private dataset holding `pipeline/` (recipe below).

```sh
rm -rf kaggle_upload && mkdir -p kaggle_upload/pipeline
cp pipeline/__init__.py pipeline/eligibility.py \
   pipeline/labeling.py pipeline/score_finbert.py  kaggle_upload/pipeline/
```

No `db.py` and no secrets this time — this run never touches the database.


In [ ]:
# --- config: the only cell you normally edit -------------------------------

CODE_DIR = "/kaggle/input/pnc-scoring"  # the dataset holding pipeline/
MODEL_DIR = "/kaggle/input/finbert-ft-2026-08-09"  # the weights
CSV_DIR = "/kaggle/input/gkg-backfill-2020-2024"  # gkg_2020.csv .. gkg_2024.csv

# Rows per forward pass. The DB notebook uses 2000 per *transaction* because
# the network round trip dominates there; here the only cost is GPU memory,
# and 256 title-length rows at MAX_LENGTH=256 fill a P100/T4 without risking
# OOM, so the GPU stays busy and a bigger number would not go faster.
BATCH = 256

OUT_PATH = "/kaggle/working/scores_gkg_2020_2024.parquet"
WORK_DIR = "/kaggle/working/run"


In [ ]:
import os
import shutil

attached = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []
for path in (CODE_DIR, MODEL_DIR, CSV_DIR):
    assert os.path.isdir(path), f"{path} not attached. Attached: {attached}"

# /kaggle/input is read-only and the package has to be importable from cwd.
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_DIR, WORK_DIR)
os.chdir(WORK_DIR)

absent = [
    p
    for p in (
        "pipeline/__init__.py",
        "pipeline/eligibility.py",
        "pipeline/labeling.py",
        "pipeline/score_finbert.py",
    )
    if not os.path.exists(p)
]
assert not absent, f"missing from the upload: {absent}"
print("staged:", WORK_DIR)


In [ ]:
# Fail here, before loading the model, if any input is wrong.
import glob
import json
import sys
import types

# score_finbert imports pipeline.db at module scope for its DB entry point.
# This run has no database, so satisfy the import with an empty stub rather
# than shipping db.py (which would drag psycopg in for nothing).
sys.modules["pipeline.db"] = types.ModuleType("pipeline.db")

import pandas as pd

from pipeline import eligibility
from pipeline.labeling import LABELS
from pipeline.score_finbert import MAX_LENGTH, load_model, predict

with open(f"{MODEL_DIR}/metrics.json", encoding="utf-8") as f:
    MODEL_VERSION = json.load(f)["model_version"]
print("model_version:", MODEL_VERSION)
print("max_length:", MAX_LENGTH, "| labels:", LABELS)

csv_paths = sorted(glob.glob(f"{CSV_DIR}/gkg_*.csv"))
assert csv_paths, f"no gkg_*.csv under {CSV_DIR}"
print("csvs:", [os.path.basename(p) for p in csv_paths])

# keep_default_na=False: a title like "NA" is text, not a missing value.
df = pd.concat(
    (pd.read_csv(p, dtype=str, keep_default_na=False) for p in csv_paths),
    ignore_index=True,
)
expected = ["bank_id", "published_at", "title", "url", "language", "attributed"]
assert list(df.columns) == expected, list(df.columns)

# The same filter serving applies: shape each row the way the gdelt
# eligibility adapter reads it and let pipeline.eligibility.check decide.
checks = [
    eligibility.check({"source": "gdelt", "title": t, "meta": {"language": lang}})
    for t, lang in zip(df["title"], df["language"])
]
df["text"] = [c.text for c in checks]
elig = df[[c.eligible for c in checks]].reset_index(drop=True)
print(f"funnel so far: {len(df)} total -> {len(elig)} eligible "
      f"({len(df) - len(elig)} ineligible, not written)")


In [ ]:
import time

torch, tokenizer, model, id2label = load_model(MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("device:", device)


# predict() builds its encoding wherever the tokenizer puts it; hand it one
# whose output follows the model so the module's logic runs unmodified on GPU.
def device_tokenizer(*args, **kwargs):
    return tokenizer(*args, **kwargs).to(device)


started = time.monotonic()
labels, probs = [], []
for lo in range(0, len(elig), BATCH):
    texts = elig["text"].iloc[lo : lo + BATCH].tolist()
    for label, dist in predict(torch, device_tokenizer, model, id2label, texts):
        labels.append(label)
        probs.append(dist)
    if (lo // BATCH) % 100 == 0:
        done = min(lo + BATCH, len(elig))
        print(f"  {done}/{len(elig)}  {time.monotonic() - started:.0f}s")

out = pd.DataFrame(
    {
        "bank_id": elig["bank_id"],
        "published_at": pd.to_datetime(elig["published_at"], utc=True),
        "title": elig["title"],
        "attributed": elig["attributed"] == "true",
        "label": labels,
        "p_negative": [d["negative"] for d in probs],
        "p_neutral": [d["neutral"] for d in probs],
        "p_positive": [d["positive"] for d in probs],
        "model_version": MODEL_VERSION,
    }
)
out.to_parquet(OUT_PATH, index=False)  # pyarrow is pandas' engine on Kaggle
print(f"funnel: {len(df)} total -> {len(elig)} eligible -> {len(out)} scored")
print(f"wrote {OUT_PATH} in {time.monotonic() - started:.0f}s")


In [ ]:
# What landed. Download scores_gkg_2020_2024.parquet from /kaggle/working.
chk = pd.read_parquet(OUT_PATH)
assert list(chk.columns) == [
    "bank_id", "published_at", "title", "attributed", "label",
    "p_negative", "p_neutral", "p_positive", "model_version",
], list(chk.columns)
assert chk["label"].isin(LABELS).all()
assert (chk["model_version"] == MODEL_VERSION).all()

print("rows:", len(chk))
print("attributed:", int(chk["attributed"].sum()),
      "| unattributed:", int((~chk["attributed"]).sum()))
dist = chk["label"].value_counts()
total = len(chk) or 1
for label in LABELS:
    print(f"  {label:9} {dist.get(label, 0):7}  {dist.get(label, 0) / total:6.1%}")

print("\ndirectional samples:")
for _, r in chk[chk["label"] != "neutral"].head(5).iterrows():
    print(f"  {r['label']:9} {r['title'][:66]}")


## Reading the class distribution

Expect it to look *more* neutral than the corpus truly is — the model
under-calls direction, which is why the parquet keeps the full
`p_negative/p_neutral/p_positive` distribution: the backtest is meant to
lower the directional threshold rather than take `label` at face value.
Unattributed rows are in here on purpose, so the attribution gate itself can
be tuned without re-scoring.

See `scoring/DESIGN.md`, "Training on a champion that missed the criteria".
